## Agente Auditor LLM — Agent A
### Auditoría con Razonamiento de Modelo de Lenguaje

Este notebook ejecuta el flujo del Agente Auditor usando un LLM como motor de razonamiento.  
A diferencia de `auditor.ipynb`, el modelo lee cada caso completo y emite directamente el diagnóstico.

**Motor:** Gemini API (`gemini-2.5-flash`) — requiere `GEMINI_API_KEY`

### 0. Instalación de dependencias

In [ ]:
# Descargar el repositorio directamente desde GitHub
!git clone https://github.com/HABalyze/agente_auditor.git

# Cambiar el directorio de trabajo a la carpeta del proyecto
%cd agente_auditor

In [ ]:
!pip install google-genai

### 1. Importaciones y designación de Rutas

In [ ]:
import json
import re
import os
import time
from pathlib import Path
from getpass import getpass

# Solicitar API key de forma segura
os.environ["GEMINI_API_KEY"] = getpass("Ingresa tu GEMINI_API_KEY: ")

from google import genai as google_genai

BASE_DIR    = Path(".")
CASOS_PATH  = BASE_DIR / "data"   / "casos.json"
REGLAS_PATH = BASE_DIR / "config" / "reglas.json"

GEMINI_MODELO = "gemini-2.5-flash"
print("Importaciones OK")

### 2. Carga de Archivos
Se lee `casos.json` con los casos de Agente B y `reglas.json` con los controles de auditoría.

In [ ]:
def cargar_json(ruta: Path, nombre: str):
    with open(ruta, "r", encoding="utf-8") as f:
        return json.load(f)

casos  = cargar_json(CASOS_PATH,  "casos.json")
reglas = cargar_json(REGLAS_PATH, "reglas.json")

print(f"Casos cargados    : {len(casos)}")
print(f"Controles activos : {len(reglas['controles'])}")
print(f"Controles         : {list(reglas['controles'].keys())}")

#### Inspección preliminar de los casos

In [ ]:
for caso in casos:
    print(f"\n{'='*60}")
    print(f"Caso {caso['id_caso']}")
    print(f"Contexto RAG : {caso['contexto_rag']}")
    print(f"Respuesta B  : {caso['respuesta_agent_b']}")

### 3. Construcción del Prompt
Se arma el prompt con el contexto del caso y las reglas de negocio.  
El LLM debe responder únicamente en JSON para poder parsearlo.

In [ ]:
def construir_prompt(caso: dict, reglas: dict) -> str:
    # Armar PROMPT con contexto del caso y reglas de negocio
    # LLM debera responder SOLO en JSON para parsearlo
    controles_resumen = []
    for nombre, ctrl in reglas["controles"].items():
        controles_resumen.append(f"- {nombre}: {ctrl['descripcion']}")

    estados = reglas["estados_posibles"]

    prompt = f"""
Eres un agente auditor de una aseguradora. Tu tarea es evaluar si Agente B tomo una decision correcta.

CONTEXTO RAG:
{caso['contexto_rag']}

RESPUESTA DE AGENTE B:
{caso['respuesta_agent_b']}

CONTROLES DE NEGOCIO APLICABLES:
{chr(10).join(controles_resumen)}

ESTADOS POSIBLES:
- APROBADO: {estados['APROBADO']}
- RECHAZADO: {estados['RECHAZADO']}
- BLOQUEADO: {estados['BLOQUEADO']}

Analiza el caso y responde UNICAMENTE con un JSON con este formato exacto, sin texto adicional:
{{"estado": "APROBADO|RECHAZADO|BLOQUEADO", "if_score": 0.00, "controles_fallidos": ["nombre_control"], "diagnostico": "Explicacion clara de la decision tomada."}}

Reglas para el if_score:
- Entre 0.75 y 1.0 si la respuesta es coherente con el contexto
- Entre 0.50 y 0.74 si hay elementos que requieren revision
- Menor a 0.50 si la respuesta contradice el contexto
""".strip()

    return prompt

# Demo — ver el prompt del Caso 1
print(construir_prompt(casos[0], reglas))

### 4. Razonador Gemini
Función que envía el prompt al modelo y retorna la respuesta en texto.

In [ ]:
def razonar_con_gemini(prompt: str) -> str:
    # Enviar prompt a Gemini API y retornara respuesta en texto
    client = google_genai.Client(
        api_key=os.getenv("GEMINI_API_KEY"),
        http_options={"api_version": "v1"}
    )
    try:
        # CORRECCIÓN: Pasar el modelo directamente como string
        respuesta = client.models.generate_content(
            model=GEMINI_MODELO,
            contents=prompt
        )
        return respuesta.text
    except Exception as e:
        print(f"[ERROR] Gemini falló: {e}")
        return ""


def parsear_respuesta_llm(texto: str) -> dict:
    # Extraer solo el bloque JSON de la respuesta
    texto = texto.strip()
    texto = re.sub(r"```json|```", "", texto).strip()
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", texto, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                pass
    return {
        "estado": "RECHAZADO",
        "if_score": 0.0,
        "controles_fallidos": [],
        "diagnostico": f"No se pudo parsear la respuesta del LLM: {texto[:200]}"
    }

print("Razonador Gemini listo")

### 5. IF — Indice de Fidelidad Analtica
En esta versión el LLM emite el `if_score` como parte de su razonamiento.  
Solo clasificamos el valor según los umbrales del `reglas.json`.

In [ ]:
def clasificar_if(score: float, umbrales: dict) -> str:
    if score >= umbrales["CONFORME"]:
        return "CONFORME"
    return "NO_CONFORME"


print("Clasificador IF listo")

### 6. Motor de Diagnóstico
Orquestación del flujo completo — el LLM reemplaza `evaluar_control` y `diagnosticar_caso`.

In [ ]:
def diagnosticar_caso(caso: dict, reglas: dict) -> dict:
    # El LLM lee el caso completo y las reglas, y emite el diagnóstico

    prompt        = construir_prompt(caso, reglas)
    respuesta_llm = razonar_con_gemini(prompt)
    resultado_llm = parsear_respuesta_llm(respuesta_llm)

    # Clasificar IF según umbrales del reglas.json
    if_score     = float(resultado_llm.get("if_score", 0.0))
    if_categoria = clasificar_if(if_score, reglas["umbrales"]["if"])

    return {
        "id_caso":            caso["id_caso"],
        "estado":             resultado_llm.get("estado", "OBSERVACION"),
        "if_score":           round(if_score, 4),
        "if_categoria":       if_categoria,
        "diagnostico":        resultado_llm.get("diagnostico", "Sin diagnóstico."),
        "controles_fallidos": resultado_llm.get("controles_fallidos", []),
    }

print("Motor de diagnóstico listo")

### 7. Ejecución de Auditoría Completa
Se procesan los 4 casos y se genera el diagnóstico en la estructura requerida.

In [ ]:
SEP = "─" * 60
resultados = []

for caso in casos:
    print(f"      Analizando caso {caso['id_caso']}...")
    resultado = diagnosticar_caso(caso, reglas)
    resultados.append(resultado)

    print(SEP)
    print(f"Caso {resultado['id_caso']}: {resultado['estado']}")
    print(f"- Índice de Fidelidad Analítica: {resultado['if_score']} ({resultado['if_categoria']})")
    print(f"- Diagnóstico/Razón: {resultado['diagnostico']}")

    # PASO 2: Pausa la ejecución por 12 segundos antes del siguiente caso
    # print("      (Pausa de 12s para respetar la cuota de Google AI Studio...)")
    time.sleep(30)

print(SEP)

### 8. Resumen de Auditoría

In [ ]:
print("\n── RESUMEN DE AUDITORÍA ──────────────────────")
estados = [r["estado"] for r in resultados]
for estado in ["APROBADO", "RECHAZADO", "BLOQUEADO"]:
    count = estados.count(estado)
    if count:
        print(f"  {estado}: {count} caso(s)")
print(SEP)